# 01 · Cleaning the Hertfordshire collision data

**Project:** Hertfordshire road collision hotspots (2021–2025)
**Source:** Department for Transport, STATS19 road safety open data ([gov.uk](https://www.gov.uk/government/statistical-data-sets/road-safety-open-data)), Open Government Licence.

The raw files store everything as number codes (for example `collision_severity = 2` means *Serious*). This notebook:

1. Loads the Hertfordshire extract made by `scripts/01_filter_herts.py`
2. Turns codes into readable labels using the DfT data guide
3. Marks missing values properly (DfT uses `-1`)
4. Builds date/time fields and useful flags (KSI, darkness, weekend, road name…)
5. Summarises vehicles and casualties per collision
6. Runs data quality checks and saves clean files to `data/clean/`

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, CLEAN = ROOT / "data/raw", ROOT / "data/clean"

pd.set_option("display.max_columns", 60)

collisions = pd.read_csv(CLEAN / "herts_collisions_raw_codes.csv", low_memory=False)
vehicles   = pd.read_csv(CLEAN / "herts_vehicles_raw_codes.csv", low_memory=False)
casualties = pd.read_csv(CLEAN / "herts_casualties_raw_codes.csv", low_memory=False)

for name, df in [("collisions", collisions), ("vehicles", vehicles), ("casualties", casualties)]:
    print(f"{name:11s} {df.shape[0]:>6,} rows × {df.shape[1]} columns")

collisions   8,375 rows × 44 columns
vehicles    16,307 rows × 32 columns
casualties  11,454 rows × 24 columns


## 1. Drop columns we don't need

- `*_historic` columns use the old (pre-2024) coding and are mostly empty for recent years. DfT provides the new-format column alongside each one.
- `local_authority_district` is retired (all `-1`); `local_authority_ons_district` replaces it.
- `police_force` is always 41 (Hertfordshire).

In [2]:
def drop_unused(df):
    cols = [c for c in df.columns if c.endswith("_historic")]
    cols += [c for c in ["local_authority_district", "police_force"] if c in df.columns]
    return df.drop(columns=cols)

collisions, vehicles, casualties = (drop_unused(d) for d in (collisions, vehicles, casualties))
print(collisions.shape, vehicles.shape, casualties.shape)

(8375, 38) (16307, 29) (11454, 24)


## 2. Decode labels from the DfT data guide

The guide lists every `(table, field, code) → label`. We keep the original number column and add a readable `_label` version, which makes filtering easy later.

Some fields look coded but are really numbers (ages, engine size, road number), so we leave those as numbers.

In [3]:
guide = pd.read_excel(RAW / "data-guide-2025.xlsx", sheet_name="2024_code_list")
guide.columns = ["table", "field", "code", "label", "note"]
guide = guide.dropna(subset=["code", "label"])
guide["code"] = guide["code"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

KEEP_AS_NUMBER = {"age_of_driver", "age_of_casualty", "engine_capacity_cc", "first_road_number",
                  "second_road_number", "age_of_vehicle", "speed_limit"}
SKIP = {"date", "time", "generic_make_model", "local_authority_highway", "local_authority_highway_current"}

def decode(df, table):
    df = df.copy()
    lookup = guide[guide.table == table]
    decoded = []
    for field, g in lookup.groupby("field"):
        if field not in df.columns or field in KEEP_AS_NUMBER | SKIP:
            continue
        mapping = dict(zip(g.code, g.label.str.strip()))
        df[f"{field}_label"] = df[field].astype(str).map(mapping)
        decoded.append(field)
    return df, decoded

collisions, dec_c = decode(collisions, "collision")
vehicles,   dec_v = decode(vehicles, "vehicle")
casualties, dec_k = decode(casualties, "casualty")
print(f"Decoded {len(dec_c)} collision, {len(dec_v)} vehicle, {len(dec_k)} casualty fields")

Decoded 18 collision, 20 vehicle, 15 casualty fields


### Check: any codes the guide didn't cover?
If a label is missing where the code exists, we'd silently lose information — so we check.

In [4]:
def unmapped(df, fields):
    rows = []
    for f in fields:
        miss = df[df[f"{f}_label"].isna()][f]
        if len(miss):
            rows.append((f, len(miss), sorted(miss.unique().tolist())[:8]))
    return pd.DataFrame(rows, columns=["field", "rows_unmapped", "codes"])

pd.concat([unmapped(collisions, dec_c).assign(table="collision"),
           unmapped(vehicles, dec_v).assign(table="vehicle"),
           unmapped(casualties, dec_k).assign(table="casualty")])

,field,rows_unmapped,codes,table
0,carriageway_hazards,7447,[0],collision
1,special_conditions_at_site,4665,[0],collision
0,driver_distance_banding,11026,[-1],vehicle
1,hit_object_in_carriageway,14488,[0],vehicle
2,hit_object_off_carriageway,15623,[0],vehicle
3,skidding_and_overturning,13670,[0],vehicle
0,casualty_type,68,[33],casualty


**Fixing the gaps found above:**
- Code `0` in the hazard, skid and "hit object" fields means *None*. The guide leaves it out.
- Casualty type `33` is *Personal powered transporter* (mostly e-scooters). The guide lists it under 23 by mistake; vehicle type 33 has the same meaning.
- `driver_distance_banding = -1` is genuinely missing and becomes "Unknown" in the next step.

In [5]:
ZERO_MEANS_NONE = ["carriageway_hazards", "special_conditions_at_site", "hit_object_in_carriageway",
                   "hit_object_off_carriageway", "skidding_and_overturning"]
for df in (collisions, vehicles):
    for f in ZERO_MEANS_NONE:
        if f in df.columns:
            df.loc[df[f].eq(0), f"{f}_label"] = "None"
casualties.loc[casualties["casualty_type"].eq(33), "casualty_type_label"] = "Personal powered transporter (e.g. e-scooter)"

pd.concat([unmapped(collisions, dec_c), unmapped(vehicles, dec_v), unmapped(casualties, dec_k)])

,field,rows_unmapped,codes
0,driver_distance_banding,11026,[-1]


## 3. Missing values

DfT writes `-1` for *data missing or out of range*. In the number columns we turn that into a proper `NaN` so averages aren't dragged down by fake -1s. In label columns these become `"Unknown"`.

Speed limit also uses `99` for *unknown (self reported)*.

In [6]:
for df in (collisions, vehicles, casualties):
    num_cols = [c for c in df.columns if c in KEEP_AS_NUMBER or c in ("enhanced_severity_collision",)]
    for c in num_cols:
        df[c] = df[c].where(df[c] >= 0)
    for c in [c for c in df.columns if c.endswith("_label")]:
        df[c] = df[c].replace({"Data missing or out of range": "Unknown"}).fillna("Unknown")

collisions["speed_limit"] = collisions["speed_limit"].where(collisions["speed_limit"] != 99)

missing = (collisions.isna().mean() * 100).round(1)
missing[missing > 0].sort_values(ascending=False).to_frame("% missing")

,% missing
second_road_number,38.4


## 4. Dates, times and location fields

In [7]:
c = collisions
c["datetime"] = pd.to_datetime(c["date"] + " " + c["time"], format="%d/%m/%Y %H:%M", errors="coerce")
c["date"]  = pd.to_datetime(c["date"], format="%d/%m/%Y")
c["year"]  = c["date"].dt.year
c["month"] = c["date"].dt.month
c["month_name"] = c["date"].dt.month_name().str[:3]
c["hour"]  = c["datetime"].dt.hour
c["weekday"] = c["date"].dt.day_name()
c["is_weekend"] = c["weekday"].isin(["Saturday", "Sunday"])

# Commuting peaks: weekdays 07:00–09:59 and 16:00–18:59
c["time_band"] = pd.cut(c["hour"], bins=[-1, 5, 9, 15, 18, 21, 23],
                        labels=["Night (00–05)", "Morning peak (06–09)", "Daytime (10–15)",
                                "Evening peak (16–18)", "Evening (19–21)", "Late (22–23)"])

c["district"] = c["local_authority_ons_district_label"]

# Road name such as "A414", "M25", or "Unclassified"
prefix = {1: "M", 2: "A", 3: "A", 4: "B", 5: "C"}
def road_name(cls, num):
    if cls in prefix and pd.notna(num) and num > 0:
        name = f"{prefix[cls]}{int(num)}"
        return name + "(M)" if cls == 2 else name
    return "Unclassified" if cls == 6 else "Unknown"
c["road_name"] = [road_name(a, b) for a, b in zip(c["first_road_class"], c["first_road_number"])]

print("Bad timestamps:", c["datetime"].isna().sum())
c[["date", "time", "hour", "weekday", "time_band", "district", "road_name"]].head()

Bad timestamps: 0


,date,time,hour,weekday,time_band,district,road_name
0,2024-02-13,09:55,9,Tuesday,Morning peak (06–09),North Hertfordshire,Unclassified
1,2024-09-16,06:57,6,Monday,Morning peak (06–09),Stevenage,A1(M)
2,2024-11-03,13:15,13,Sunday,Daytime (10–15),East Hertfordshire,B1004
3,2024-12-02,11:18,11,Monday,Daytime (10–15),Hertsmere,Unclassified
4,2024-12-20,07:54,7,Friday,Morning peak (06–09),North Hertfordshire,Unclassified


## 5. Severity flags

- **KSI** (*Killed or Seriously Injured*) is the standard UK road safety measure. It's what councils and the DfT report on.
- **Adjusted severity:** police forces changed how they record injuries (the CRASH system), which made "serious" injuries look more common. DfT supplies a probability that each slight/serious collision *would have been* serious under the new system. Using it keeps comparisons fair. Hertfordshire has been on CRASH throughout 2021–2025, but we keep the adjusted figure for rigour.

In [8]:
c["severity"] = pd.Categorical(c["collision_severity_label"], ["Fatal", "Serious", "Slight"], ordered=True)
c["is_ksi"] = c["collision_severity"].isin([1, 2])
c["ksi_adjusted"] = np.where(c["collision_severity"] == 1, 1.0, c["collision_adjusted_severity_serious"].fillna(0))

c["is_dark"] = c["light_conditions_label"].str.contains("Darkness", na=False)
c["is_wet_or_icy"] = ~c["road_surface_conditions_label"].isin(["Dry", "Unknown"])
c["is_bad_weather"] = ~c["weather_conditions_label"].str.contains("Fine no high winds|Unknown", na=False)
c["is_junction"] = c["junction_detail"].fillna(-1).gt(0)
c["is_urban"] = c["urban_or_rural_area_label"].eq("Urban")

c[["severity", "is_ksi", "ksi_adjusted"]].groupby("severity", observed=True).agg(collisions=("is_ksi", "size"), mean_adj=("ksi_adjusted", "mean"))

,collisions,mean_adj
severity,,
Fatal,116,1.000000
Serious,1638,1.000000
Slight,6621,0.000003


## 6. Who was involved? (vehicle + casualty summaries per collision)

Each collision has several vehicles and casualties. We roll them up into one row per collision with flags like *a pedestrian was hurt* or *a young driver was involved*. This lets us answer "who" questions in the main collision table.

In [9]:
v = vehicles
vt = v["vehicle_type"]
v["is_cycle"] = vt.eq(1)
v["is_motorcycle"] = vt.isin([2, 3, 4, 5, 23, 97])
v["is_hgv"] = vt.isin([20, 21, 98])
v["is_van"] = vt.eq(19)
v["is_bus"] = vt.isin([10, 11])
v["is_escooter"] = v["escooter_flag"].eq(1)
v["young_driver"] = v["age_of_driver"].between(17, 24)
v["older_driver"] = v["age_of_driver"].ge(70)

veh_summary = v.groupby("collision_index").agg(
    involves_cycle=("is_cycle", "any"), involves_motorcycle=("is_motorcycle", "any"),
    involves_hgv=("is_hgv", "any"), involves_van=("is_van", "any"),
    involves_bus=("is_bus", "any"), involves_escooter=("is_escooter", "any"),
    involves_young_driver=("young_driver", "any"), involves_older_driver=("older_driver", "any"),
)

k = casualties
ct = k["casualty_type"]
k["casualty_group"] = np.select(
    [ct.eq(0), ct.eq(1), ct.isin([2, 3, 4, 5, 23, 97]), ct.isin([8, 9]), ct.isin([19, 20, 21, 98]),
     ct.isin([10, 11]), ct.eq(33)],
    ["Pedestrian", "Cyclist", "Motorcyclist", "Car occupant", "Goods vehicle occupant",
     "Bus/minibus occupant", "E-scooter / PPT user"],
    default="Other")
k["severity"] = pd.Categorical(k["casualty_severity_label"], ["Fatal", "Serious", "Slight"], ordered=True)
k["is_ksi"] = k["casualty_severity"].isin([1, 2])

cas_summary = k.groupby("collision_index").agg(
    pedestrian_casualties=("casualty_group", lambda s: (s == "Pedestrian").sum()),
    cyclist_casualties=("casualty_group", lambda s: (s == "Cyclist").sum()),
    motorcyclist_casualties=("casualty_group", lambda s: (s == "Motorcyclist").sum()),
    ksi_casualties=("is_ksi", "sum"),
    child_casualties=("age_of_casualty", lambda s: s.lt(16).sum()),
)

c = c.join(veh_summary, on="collision_index").join(cas_summary, on="collision_index")
flag_cols = list(veh_summary.columns)
c[flag_cols] = c[flag_cols].fillna(False).astype(bool)
c[list(cas_summary.columns)] = c[list(cas_summary.columns)].fillna(0).astype(int)
c["vulnerable_user_hurt"] = (c[["pedestrian_casualties", "cyclist_casualties", "motorcyclist_casualties"]].sum(axis=1) > 0)

k["casualty_group"].value_counts().to_frame("casualties")

,casualties
casualty_group,
Car occupant,7463
Pedestrian,1190
Motorcyclist,1145
Cyclist,918
Goods vehicle occupant,535
Bus/minibus occupant,72
E-scooter / PPT user,68
Other,63


## 7. Data quality checks
Simple tests that would fail loudly if something went wrong above.

In [10]:
checks = {
    "collision_index unique": c["collision_index"].is_unique,
    "every vehicle links to a collision": vehicles["collision_index"].isin(c["collision_index"]).all(),
    "every casualty links to a collision": casualties["collision_index"].isin(c["collision_index"]).all(),
    "casualty counts match": (k.groupby("collision_index").size().reindex(c["collision_index"]).fillna(0).values
                              == c["number_of_casualties"].values).all(),
    "vehicle counts match": (v.groupby("collision_index").size().reindex(c["collision_index"]).fillna(0).values
                             == c["number_of_vehicles"].values).all(),
    "all dates 2021–2025": c["year"].between(2021, 2025).all(),
    "all 10 districts present": c["district"].nunique() == 10,
    "coordinates inside Herts box": c.dropna(subset=["latitude"]).eval(
        "51.55 < latitude < 52.10 and -0.80 < longitude < 0.25").all(),
}
pd.Series(checks, name="passed").to_frame()

,passed
collision_index unique,True
every vehicle links to a collision,True
every casualty links to a collision,True
casualty counts match,True
vehicle counts match,True
all dates 2021–2025,True
all 10 districts present,True
coordinates inside Herts box,True


In [11]:
missing_coords = c[c["latitude"].isna()]
print(f"Collisions without coordinates: {len(missing_coords)} (kept in tables, excluded from maps)")
missing_coords[["collision_index", "date", "district", "road_name", "severity"]]

Collisions without coordinates: 1 (kept in tables, excluded from maps)


,collision_index,date,district,road_name,severity
4422,2022411239270,2022-11-01,East Hertfordshire,A1,Slight


## 8. Save clean data
Parquet keeps the column types (dates, categories, booleans), and it's smaller and faster than CSV. We also save a CSV of collisions so it opens in Excel, Power BI or Tableau.

In [12]:
c = c.sort_values("datetime").reset_index(drop=True)
c.to_parquet(CLEAN / "collisions.parquet", index=False)
v.to_parquet(CLEAN / "vehicles.parquet", index=False)
k.to_parquet(CLEAN / "casualties.parquet", index=False)

bi_cols = ["collision_index", "datetime", "date", "year", "month", "month_name", "weekday", "hour", "time_band",
           "district", "road_name", "latitude", "longitude", "severity", "is_ksi", "ksi_adjusted",
           "number_of_vehicles", "number_of_casualties", "speed_limit", "road_type_label",
           "junction_detail_label", "light_conditions_label", "weather_conditions_label",
           "road_surface_conditions_label", "urban_or_rural_area_label", "is_dark", "is_wet_or_icy",
           "is_junction", *flag_cols, *cas_summary.columns, "vulnerable_user_hurt", "lsoa_of_accident_location"]
c[bi_cols].to_csv(CLEAN / "collisions_for_bi.csv", index=False)

for f in sorted(CLEAN.glob("*")):
    print(f"{f.name:40s} {f.stat().st_size/1024:8.0f} KB")

casualties.parquet                            277 KB
collisions.parquet                            683 KB
collisions_for_bi.csv                        2710 KB
herts_casualties_raw_codes.csv                947 KB
herts_collisions_raw_codes.csv               1614 KB
herts_vehicles_raw_codes.csv                 1783 KB
vehicles.parquet                              484 KB


## 9. First look: headline numbers

In [13]:
summary = c.groupby("year").agg(
    collisions=("collision_index", "size"),
    fatal=("collision_severity", lambda s: (s == 1).sum()),
    serious=("collision_severity", lambda s: (s == 2).sum()),
    ksi=("is_ksi", "sum"),
    casualties=("number_of_casualties", "sum"),
)
summary["ksi_%"] = (summary["ksi"] / summary["collisions"] * 100).round(1)
summary

,collisions,fatal,serious,ksi,casualties,ksi_%
year,,,,,,
2021,1598,19,323,342,2161,21.4
2022,1780,24,339,363,2518,20.4
2023,1813,29,357,386,2416,21.3
2024,1648,21,294,315,2267,19.1
2025,1536,23,325,348,2092,22.7


In [14]:
(c.groupby("district")
  .agg(collisions=("collision_index", "size"), ksi=("is_ksi", "sum"),
       pedestrians_hurt=("pedestrian_casualties", "sum"), cyclists_hurt=("cyclist_casualties", "sum"))
  .assign(ksi_pct=lambda d: (d.ksi / d.collisions * 100).round(1))
  .sort_values("collisions", ascending=False))

,collisions,ksi,pedestrians_hurt,cyclists_hurt,ksi_pct
district,,,,,
St Albans,1268,254,129,151,20.0
Hertsmere,1182,228,154,90,19.3
East Hertfordshire,917,203,143,94,22.1
Dacorum,872,202,148,112,23.2
North Hertfordshire,845,213,113,113,25.2
Welwyn Hatfield,815,184,107,104,22.6
Broxbourne,691,121,116,67,17.5
Watford,658,126,142,93,19.1
Three Rivers,592,128,62,49,21.6


In [15]:
c["road_name"].value_counts().head(12).to_frame("collisions")

,collisions
road_name,
Unclassified,3281
M25,389
M1,384
A414,297
A10,239
A1(M),215
A602,210
A41,188
A5183,182


## Summary

The data is now clean, labelled and checked. Saved files:

| File | What's in it |
|---|---|
| `collisions.parquet` | One row per collision, readable labels, time fields, severity and "who was involved" flags |
| `vehicles.parquet` | One row per vehicle, with vehicle type flags |
| `casualties.parquet` | One row per casualty, with casualty group |
| `collisions_for_bi.csv` | A tidy subset for Excel / Power BI / Tableau |

**Next:** `02_eda.ipynb`, the exploratory analysis of when, where, who and under what conditions.